In [ ]:
!pip install sktime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.6/37.6 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 8.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

from model_classes import W4_Regression

In [ ]:
# Preset Values

filename = "CRMLS_0625-0626_clean.csv"

end_yr = 2026
end_mnth = 6


main_cols = ["BedroomsTotal", "BathroomsTotalInteger", "LivingArea", "LotSizeSquareFeet",
             "DaysOnMarket", "YearBuilt", "PostalCode", "SaleMonth"]

extras = ["ViewYN", "FireplaceYN", "NewConstructionYN", "PoolPrivateYN"]
extra_cols = [a+"_True" for a in extras] + [a+"_False" for a in extras]

totals = main_cols+extra_cols

target = "ClosePrice"

In [ ]:
def load_df(file=filename):
  """
  Takes a .csv filename or filepath
  Returns the DataFrame loaded from the csv
  """
  df = pd.read_csv(file, low_memory=False)
  df["CloseDate"] = pd.to_datetime(df["CloseDate"])     # Converts "CloseDate" values to datetime type
  return df

In [ ]:
main_df = load_df()

In [ ]:
print(f"Column Check: \t {main_df.shape}")

In [ ]:
class W5_Regs():

  def __init__(self,df):
    self.df = df


  def test_train_split(self, yr=end_yr, mo=end_mnth):
    """
    Takes a DataFrame
    Encodes "PropertyType" column
    Returns a defined training and test set for the DataFrame
    """
    yr_mo = []
    for i in self.df['CloseDate']:                                         # For each date in the "CloseDate" column
      yr, mo = i.year, i.month                                          # Define the year and month values of date i
      yr_mo.append([yr,mo])                                             # Append to "yr_mo" a list of date i's year and month
    te_set = [b for b in range(len(yr_mo)) if yr_mo[b] == [yr,mo]]   # Define a list of row #s with date 06/2026
    te_rng = te_set[0::len(te_set)-1]                                 # Define a list of the first and last row in "te_set"
    tr, te = self.df[0:te_rng[0]], self.df[te_rng[0]:te_rng[1]]             # Define the training and test sets of the inputted df
    return tr, te


  def TreeReg(self, tr, te, feat=totals, targ=target):
    """
    Takes a training DataFrame and test DataFrame
    """
    x_tr = tr[feat].values            # Define the x_train set values
    y_tr = tr[targ].values            # Define the y_train set values
    x_te = te[feat].values            # Define the x_test set values
    y_te = te[targ].values            # Define the y_test set values

    model = DecisionTreeRegressor(random_state=0)        # Define Decision Tree Regression model
    model.fit(x_tr, y_tr)             # Fit training data to model
    y_pred = model.predict(x_te)      # Models the predicted y values from x_test values
    return y_te, y_pred


  def ForestReg(self, tr, te, feat=totals, targ=target):
    """
    Takes a training DataFrame and test DataFrame
    """
    x_tr = tr[feat].values            # Define the x_train set values
    y_tr = tr[targ].values            # Define the y_train set values
    x_te = te[feat].values            # Define the x_test set values
    y_te = te[targ].values            # Define the y_test set values

    model = RandomForestRegressor(random_state=0)        # Define Random Forest Regression model
    model.fit(x_tr, y_tr)             # Fit training data to model
    y_pred = model.predict(x_te)      # Models the predicted y values from x_test values
    return y_te, y_pred


  def r2_eval(self, y_test, y_pred):
    r2 = r2_score(y_test, y_pred)       # Computes the r2 score of y_test and the predicted y
    return r2


  def shrt_main(self, model, trn, tes, evals, feats, rev=True, prt=True, **kwargs):

    scores = []
    for col in feats:
      try:
        y_te, y_pr = model(trn, tes, [col], dep=kwargs["dep"], l_rate=kwargs["l_rate"], est=kwargs["est"])
      except:
        try:
          y_te, y_pr = model(trn, tes, [col], dep=kwargs["dep"], l_rate=kwargs["l_rate"])
        except:
          y_te, y_pr = model(trn, tes, [col])
      scr = evals(y_te, y_pr)
      scores.append(scr)
    scr_lst = [[feats[i], scores[i]] for i in range(len(scores))]
    scr_lst = sorted(scr_lst, key=lambda x: x[1], reverse=rev)

    totals = []
    for i in range(1, len(scores)):
      maximum = i
      cols = [scr_lst[a][0] for a in range(maximum)]
      try:
        y_te, y_pr = model(trn, tes, cols, dep=kwargs["dep"], l_rate=kwargs["l_rate"], est=kwargs["est"])
      except:
        try:
          y_te, y_pr = model(trn, tes, cols, dep=kwargs["dep"], l_rate=kwargs["l_rate"])
        except:
          y_te, y_pr = model(trn, tes, cols)
      scr = evals(y_te, y_pr)
      totals.append([cols, scr])
    totals = sorted(totals, key=lambda x: x[1], reverse=rev)
    if prt == True:
      print(f"Correlation of {totals[0][0]}:\n {round(totals[0][1],4)}")

    return totals[0][0], totals[0][1]


  def lng_main(self, model, trn, tes, evals, feats, rev=True, prt=True, **kwargs):

    scores = []
    for col in feats:
      try:
        y_te, y_pr = model(trn, tes, [col], dep=kwargs["dep"], l_rate=kwargs["l_rate"], est=kwargs["est"])
      except:
        try:
          y_te, y_pr = model(trn, tes, [col], dep=kwargs["dep"], l_rate=kwargs["l_rate"])
        except:
          y_te, y_pr = model(trn, tes, [col])
      scr = evals(y_te, y_pr)
      scores.append(scr)
    scr_lst = [[feats[i], scores[i]] for i in range(len(scores))]
    scr_lst = sorted(scr_lst, key=lambda x: x[1], reverse=rev)

    totals = []
    for i in range(1, len(scores)):
      maximum = i
      cols = [scr_lst[a][0] for a in range(maximum)]
      try:
        y_te, y_pr = model(trn, tes, cols, dep=kwargs["dep"], l_rate=kwargs["l_rate"], est=kwargs["est"])
      except:
        try:
          y_te, y_pr = model(trn, tes, cols, dep=kwargs["dep"], l_rate=kwargs["l_rate"])
        except:
          y_te, y_pr = model(trn, tes, cols)
      scr = evals(y_te, y_pr)
      if prt == True:
        print(f"Correlation of {cols}: \t\t {round(scr,4)}")

      totals.append([cols, scr])
    totals = sorted(totals, key=lambda x: x[1], reverse=True)
    return totals[0][0], totals[0][1]

In [ ]:
Week5 = W5_Regs(main_df)

In [ ]:
base = W4_Regression(main_df)
train, test = base.test_train_split()

# **Decision Tree Regressor**

***Non-Transform DataFrame***

In [ ]:
if __name__ == "__main__":
  Week5.lng_main(Week5.TreeReg, train, test, Week5.r2_eval, totals)

Correlation of ['PostalCode']: 		 0.4947
Correlation of ['PostalCode', 'BathroomsTotalInteger']: 		 0.6916
Correlation of ['PostalCode', 'BathroomsTotalInteger', 'LivingArea']: 		 0.5846
Correlation of ['PostalCode', 'BathroomsTotalInteger', 'LivingArea', 'BedroomsTotal']: 		 0.6173
Correlation of ['PostalCode', 'BathroomsTotalInteger', 'LivingArea', 'BedroomsTotal', 'PoolPrivateYN_False']: 		 0.5826
Correlation of ['PostalCode', 'BathroomsTotalInteger', 'LivingArea', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_False']: 		 0.5992
Correlation of ['PostalCode', 'BathroomsTotalInteger', 'LivingArea', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True']: 		 0.6027
Correlation of ['PostalCode', 'BathroomsTotalInteger', 'LivingArea', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_True']: 		 0.5599
Correlation of ['PostalCode', 'BathroomsTotalInteger', 'LivingArea', 'BedroomsTotal', 'PoolPrivateYN_False', 'F

***Results 06/25-06/26***

Highest R2 Score:  **0.6916**

* Included Features:

  - ['PostalCode', 'BathroomsTotalInteger']

***Results 05/25-05/26***

Highest R2 Score:  **0.3905**

* Included Features:

  - ['PostalCode', 'BathroomsTotalInteger', 'BedroomsTotal', 'LivingArea', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_True', 'ViewYN_True', 'ViewYN_False', 'YearBuilt','NewConstructionYN_True', 'NewConstructionYN_False']

* Excluded Features:

  - ['SaleMonth', 'DaysOnMarket']

---

* Wide range of R2 scores for different combination of features


***Log Transform DataFrame***

In [ ]:
log_df = base.log_transform()
Wk5 = W5_Regs(log_df)
tr, te = Wk5.test_train_split()

In [ ]:
if __name__ == "__main__":
  Wk5.lng_main(Wk5.TreeReg, tr, te, Wk5.r2_eval, totals)

Correlation of ['PostalCode']: 		 0.7557
Correlation of ['PostalCode', 'LivingArea']: 		 0.8258
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger']: 		 0.8243
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal']: 		 0.8164
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'PoolPrivateYN_False']: 		 0.8176
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_False']: 		 0.8136
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True']: 		 0.8163
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_True']: 		 0.8132
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_

In [ ]:
try_cols = ['PostalCode', 'LivingArea', 'PoolPrivateYN_False', 'FireplaceYN_True']

y_te, y_pr = Wk5.TreeReg(tr, te, try_cols)
r2 = Wk5.r2_eval(y_te, y_pr)
print(f"{round(r2,4)}")

0.8282


***Results 06/25-06/26***

Highest R2 Score:  **0.8282**

* Included Features:

  - ['PostalCode', 'LivingArea']

***Results 05/25-05/26***


Highest R2 Score:  **0.7995**

* Included Features:

  - ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'ViewYN_False', 'NewConstructionYN_True', 'ViewYN_True', 'NewConstructionYN_False']

* Excluded Features:

  - ['PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_True', 'YearBuilt', 'SaleMonth', 'DaysOnMarket']

---

* All R2 scores are within a reasonable range, and do not deviate significantly from one another


# **Random Forest Regressor**

***Non-Transform DataFrame***

In [ ]:
if __name__ == "__main__":
  Week5.lng_main(Week5.ForestReg, train, test, Week5.r2_eval, totals)

Correlation of ['PostalCode']: 		 0.4945
Correlation of ['PostalCode', 'LivingArea']: 		 0.7345
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger']: 		 0.7402
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal']: 		 0.7381
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'PoolPrivateYN_False']: 		 0.737
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_False']: 		 0.7346
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True']: 		 0.7346
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_True']: 		 0.731
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_Fa

In [ ]:
try_cols = ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'YearBuilt', 'ViewYN_False', 'NewConstructionYN_True']

y_te, y_pr = Week5.ForestReg(train, test, try_cols)
r2 = Week5.r2_eval(y_te, y_pr)
print(f"{round(r2,4)}")

0.7718


**Results 06/25-06/26**

Highest R2 Score:  **0.7718**

* Included Features:

  - ['PostalCode', 'BathroomsTotalInteger', 'LivingArea', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True', 'YearBuilt', 'PoolPrivateYN_True']

**Results 05/25-05/26**

Highest R2 Score:  **0.3911**

* Included Features:

  - ['BathroomsTotalInteger', 'PostalCode', 'YearBuilt', 'SaleMonth', 'NewConstructionYN_False']

* Excluded Features:

  - ['BedroomsTotal', 'LivingArea', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_True', 'ViewYN_True', 'ViewYN_False', 'NewConstructionYN_True', 'DaysOnMarket']

---

* Less wide-ranging values of R2 scores than Decision Tree Regressor


***Log Transform DataFrame***

In [ ]:
if __name__ == "__main__":
  Wk5.lng_main(Wk5.ForestReg, tr, te, Wk5.r2_eval, totals)

Correlation of ['PostalCode']: 		 0.7566
Correlation of ['PostalCode', 'LivingArea']: 		 0.8829
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger']: 		 0.8866
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal']: 		 0.8893
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'PoolPrivateYN_False']: 		 0.8903
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_False']: 		 0.8912
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True']: 		 0.8914
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_True']: 		 0.8907
Correlation of ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_

In [ ]:
try_cols = ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True', 'LotSizeSquareFeet', 'ViewYN_False', 'ViewYN_True']

y_te, y_pr = Wk5.ForestReg(tr, te, try_cols)
r2 = Wk5.r2_eval(y_te, y_pr)
print(f"{round(r2,4)}")

0.8965


**Results 06/25-06/26**

Highest R2 Score:  **0.8965**

* Included Features:

  - ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_False', 'YearBuilt', 'PoolPrivateYN_True', 'LotSizeSquareFeet']


**Results 05/25-05/26**

Highest R2 Score:  **0.8842**

* Included Features:

  - ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_False', 'LotSizeSquareFeet', 'YearBuilt', 'PoolPrivateYN_True']

  - ['PostalCode', 'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'FireplaceYN_True', 'PoolPrivateYN_False', 'LotSizeSquareFeet', 'YearBuilt', 'PoolPrivateYN_True', 'NewConstructionYN_True']

* Excluded Features:

  - ['ViewYN_False', 'NewConstructionYN_True', 'ViewYN_True', 'NewConstructionYN_False', 'SaleMonth', 'DaysOnMarket']

  - ['ViewYN_False', FireplaceYN_False, 'ViewYN_True', 'NewConstructionYN_False', 'SaleMonth', 'DaysOnMarket']

---

* All R2 scores are within a reasonable range, and do not deviate significantly from one another
* Less wide-ranging values of R2 scores than Decision Tree Regressor


# **Week 6 - Feature Engineering**

***Features:***

**Ratios:**

* "LivingArea" to "LotSizeSquareFeet"
  - "LotSizeSquareFeet" to "DaysOnMarket"
  - "LivingArea" to "DaysOnMarket"
* "BathroomsTotalInteger" to "BedroomsTotal"

---
**Other:**

* Average "ClosePrice" per "PostalCode" (.groupby())
* Average "LotSizeSquareFeet" per "PostalCode" (.groupby())
* Average "LivingArea" per "PostalCode" (.groupby())

In [ ]:
# Ratios

combos = [["LivingArea", "LotSizeSquareFeet"], ["LotSizeSquareFeet", "DaysOnMarket"],
          ["LivingArea", "DaysOnMarket"], ["BathroomsTotalInteger", "BedroomsTotal"]]

new_names = [["LivingArea/LotSizeSquareFeet"], ["LotSizeSquareFeet/DaysOnMarket"],
              ["LivingArea/DaysOnMarket"], ["BathroomsTotalInteger/BedroomsTotal"]]

CRMLS_df = main_df.copy()
a=0
for set in combos:
  feat = [main_df[set[0]][i]/main_df[set[1]][i] for i in range(len(main_df))]
  CRMLS_df.insert(CRMLS_df.shape[1], new_names[a][0], feat)
  a += 1

In [ ]:
# Averages

def get_agg(df, col_name, new_name):
  post = df.groupby(by=["PostalCode"], as_index=False).agg({col_name: "mean"})
  enc = {post["PostalCode"].iloc[i]: round(post[col_name].iloc[i],2) for i in range(len(post))}

  vals = main_df["PostalCode"].to_list()
  lst = pd.Series(vals)
  lst.replace(enc, inplace=True)
  avgs = lst.to_list()
  df.insert(df.shape[1], new_name, avgs)

In [ ]:
get_agg(CRMLS_df, "ClosePrice", "AvgAreaCost")
get_agg(CRMLS_df, "LotSizeSquareFeet", "AvgAreaLot")
get_agg(CRMLS_df, "LivingArea", "AvgLivingArea")

# **Geographic Layer of School Districts**

* Imported GeoJSON file
---
* "Latitude" != 0
  - Also eliminated "Longitude" values equal to 0
* "DistrictType" == "Unified" (reduced rows from 937 to 345)
---
* Converted cleaned CRMLS (main_df) "Longitude", "Latitude" to Geometric Array of geographic points
* Created copy of main_df with new column "geometry" for geographic points
* Converted copy of main_df into a GeoDataFrame (geo_gdf) with "geometry" as geometric column
---
* Defined and spatially joined geo_gdf and school district gdf (schl_gdf) as combo_gdf
* Redefined combo_gdf with all main_df columns and only "DistrictName" column from schl_gdf

In [ ]:
# Preset file

geo_file = "California_School_District_Areas_2024-25.geojson"

enriched_df = "CRMLS_0625-0626_enriched.csv"

In [ ]:
def load_data(file=geo_file):
  """
  Takes a .csv filename or filepath
  Returns the DataFrame loaded from the csv
  """
  gdf = gpd.read_file(file)
  return gdf

In [ ]:
schl_gdf = load_data()

enr_df = CRMLS_df[CRMLS_df['Latitude'] != 0]
schl_gdf = schl_gdf[schl_gdf["DistrictType"] == "Unified"]
schl_gdf.reset_index(drop=True)

geo = gpd.points_from_xy(x=enr_df.Longitude, y=enr_df.Latitude, crs=schl_gdf.crs)
geo_main = enr_df.copy()
geo_main["geometry"] = geo
geo_gdf = gpd.GeoDataFrame(geo_main).set_geometry(col="geometry")

combo_gdf = gpd.sjoin(geo_gdf, schl_gdf)
combo_gdf = combo_gdf[enr_df.columns.to_list()+["DistrictName"]]
combo_gdf.reset_index(drop=True, inplace=True)

**Assigns an ID to each "DistrictName" school, and adds the IDs as a separate column**

In [ ]:
dist1 = combo_gdf.sort_values(by="PostalCode")
dist = dist1.groupby(by=["DistrictName"], sort=False, as_index=False).size()
enc = {dist["DistrictName"].iloc[i]: i+1 for i in range(len(dist))}

vals = combo_gdf["DistrictName"].to_list()
lst = pd.Series(vals)
lst.replace(enc, inplace=True)
ids = lst.to_list()
combo_gdf.insert(combo_gdf.shape[1], "DistrictID", ids)

/tmp/ipykernel_1181/727000183.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lst.replace(enc, inplace=True)


In [ ]:
def save_csv(df, file=enriched_df):
  """
  Takes a DataFrame
  Saves the inputted DataFrame as a .csv file, given inputted name
  """
  df.to_csv(file, index=False)

In [ ]:
save_csv(combo_gdf)

# ***Model Re-evalutation***

In [ ]:
print("Changes in Data Quantity")
print("------------------------ \n")

print(f"Starting length of cleaned CRMLS DataFrame: \t {len(CRMLS_df)} rows \n")

print(f"Eliminating zero-valued 'Longitude', 'Latitude': \t {len(CRMLS_df) - len(main_df)} rows dropped \n")

print(f"Length of spatially joined DataFrame: \t {len(combo_gdf)} rows")
print(f"Spatial join dropped an additional \t {len(main_df) - len(combo_gdf)} rows")

Changes in Data Quantity
------------------------ 

Starting length of cleaned CRMLS DataFrame: 	 134161 rows 

Eliminating zero-valued 'Longitude', 'Latitude': 	 0 rows dropped 

Length of spatially joined DataFrame: 	 102071 rows
Spatial join dropped an additional 	 32090 rows


In [ ]:
cols = combo_gdf.columns.to_list()
new_col_strt = [i for i in range(len(cols)) if cols[i] == "SaleMonth"]
new_cols = cols[(new_col_strt[0]+1):len(cols)]
new_cols = [i for i in new_cols if i != "DistrictName"]

crit_cols = totals+new_cols
logs = main_cols+new_cols

# **Linear Regression**

In [ ]:
Week5 = W5_Regs(combo_gdf)

In [ ]:
base = W4_Regression(combo_gdf)
train, test = base.test_train_split()

In [ ]:
regs = {"Model": [], "LogForm": [], "ScoreType": [], "ScoreValue": [], "max_depth": [], "learning_rate": [], "n_estimators": [], "columns": []}

In [ ]:
if __name__ == "__main__":
  c1, r1 = Week5.shrt_main(base.LinReg, train, test, base.r2_eval, crit_cols)
  kys,vls = list(regs.keys()), ["Linear Regression", False, "R2 Score", r1, None, None, None, c1]
  for i in range(len(kys)):
    regs[kys[i]].append(vls[i])

Correlation of ['AvgAreaCost', 'LivingArea', 'BathroomsTotalInteger', 'AvgLivingArea', 'BedroomsTotal', 'BathroomsTotalInteger/BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_True', 'PostalCode', 'LivingArea/DaysOnMarket', 'ViewYN_True', 'DistrictID', 'ViewYN_False', 'DaysOnMarket', 'AvgAreaLot', 'NewConstructionYN_True', 'YearBuilt', 'SaleMonth']:
 0.711


**Log Transform of Linear Regression**

In [ ]:
log_df = base.log_transform(feat=logs)
bs = W4_Regression(log_df)
tr, te = bs.test_train_split()

In [ ]:
if __name__ == "__main__":
  c2, r2 = Week5.shrt_main(bs.LinReg, tr, te, bs.r2_eval, crit_cols)
  kys,vls = list(regs.keys()), ["Linear Regression", True, "R2 Score", r2, None, None, None, c2]
  for i in range(len(kys)):
    regs[kys[i]].append(vls[i])

Correlation of ['AvgAreaCost', 'LivingArea', 'BathroomsTotalInteger', 'AvgLivingArea', 'BedroomsTotal', 'BathroomsTotalInteger/BedroomsTotal', 'PoolPrivateYN_False', 'LivingArea/DaysOnMarket', 'FireplaceYN_False', 'FireplaceYN_True', 'AvgAreaLot', 'LivingArea/LotSizeSquareFeet', 'DistrictID', 'PoolPrivateYN_True', 'PostalCode', 'LotSizeSquareFeet/DaysOnMarket', 'LotSizeSquareFeet', 'DaysOnMarket', 'ViewYN_False', 'NewConstructionYN_True', 'ViewYN_True', 'NewConstructionYN_False']:
 0.9014


**Results 06/25-06/26**

***Non-Transform***

*Original* Highest R2 Score:  **0.4759**

*Enriched* Highest R2 Score:  **0.711**

* Increased R2 Score

---
***Log Transform***

*Original* Highest R2 Score:  **0.5538**

*Enriched* Highest R2 Score:  **.9014**

* Increased R2 Score

# **Decision Tree Regression**

In [ ]:
if __name__ == "__main__":
  c1, r1 = Week5.shrt_main(Week5.TreeReg, train, test, Week5.r2_eval, crit_cols)
  kys,vls = list(regs.keys()), ["Decision Tree Regressor", False, "R2 Score", r1, None, None, None, c1]
  for i in range(len(kys)):
    regs[kys[i]].append(vls[i])

Correlation of ['AvgAreaCost', 'AvgLivingArea', 'AvgAreaLot', 'PostalCode', 'BathroomsTotalInteger', 'DistrictID', 'LivingArea', 'BathroomsTotalInteger/BedroomsTotal', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_True', 'YearBuilt', 'ViewYN_True', 'ViewYN_False', 'SaleMonth', 'NewConstructionYN_True', 'NewConstructionYN_False', 'DaysOnMarket', 'LotSizeSquareFeet']:
 0.709


***Log Transform DataFrame***

In [ ]:
Wk5 = W5_Regs(log_df)

In [ ]:
if __name__ == "__main__":
  c2, r2 = Wk5.shrt_main(Wk5.TreeReg, tr, te, Wk5.r2_eval, crit_cols)
  kys,vls = list(regs.keys()), ["Decision Tree Regressor", True, "R2 Score", r2, None, None, None, c2]
  for i in range(len(kys)):
    regs[kys[i]].append(vls[i])

Correlation of ['AvgAreaCost', 'PostalCode', 'AvgLivingArea', 'AvgAreaLot', 'DistrictID', 'LivingArea', 'BathroomsTotalInteger', 'BathroomsTotalInteger/BedroomsTotal', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_True', 'YearBuilt', 'ViewYN_False', 'NewConstructionYN_True', 'ViewYN_True', 'SaleMonth', 'NewConstructionYN_False', 'DaysOnMarket', 'LotSizeSquareFeet', 'LivingArea/DaysOnMarket']:
 0.8609


**Results**


***Non-Transform***

*Original* Highest R2 Score:  **0.6916**

*Enriched* Highest R2 Score:  **0.709**

 *  Drastically Increased R2 Score

---
***Log Transform***

*Original* Highest R2 Score:  **0.8282**

*Enriched* Highest R2 Score:  **0.8609**
  
 *  Increased R2 Score

# **Random Forest Regression**

In [ ]:
if __name__ == "__main__":
  c1, r1 = Week5.shrt_main(Week5.ForestReg, train, test, Week5.r2_eval, crit_cols)
  kys,vls = list(regs.keys()), ["Random Forest Regressor", False, "R2 Score", r1, None, None, None, c1]
  for i in range(len(kys)):
    regs[kys[i]].append(vls[i])

Correlation of ['AvgAreaCost', 'AvgLivingArea', 'PostalCode', 'AvgAreaLot', 'BathroomsTotalInteger', 'LivingArea', 'DistrictID', 'BathroomsTotalInteger/BedroomsTotal', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True', 'PoolPrivateYN_True', 'YearBuilt', 'ViewYN_True', 'ViewYN_False', 'SaleMonth', 'NewConstructionYN_True', 'NewConstructionYN_False', 'DaysOnMarket', 'LotSizeSquareFeet']:
 0.8099


**Log Transform DataFrame**

In [ ]:
if __name__ == "__main__":
  c2, r2 = Wk5.shrt_main(Wk5.ForestReg, tr, te, Wk5.r2_eval, crit_cols)
  kys,vls = list(regs.keys()), ["Random Forest Regressor", True, "R2 Score", r2, None, None, None, c2]
  for i in range(len(kys)):
    regs[kys[i]].append(vls[i])

Correlation of ['AvgAreaCost', 'PostalCode', 'AvgLivingArea', 'AvgAreaLot', 'DistrictID', 'LivingArea', 'BathroomsTotalInteger', 'BathroomsTotalInteger/BedroomsTotal', 'BedroomsTotal', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True', 'LotSizeSquareFeet', 'PoolPrivateYN_True', 'YearBuilt', 'ViewYN_False', 'NewConstructionYN_True', 'ViewYN_True', 'SaleMonth', 'NewConstructionYN_False', 'DaysOnMarket', 'LivingArea/DaysOnMarket', 'LotSizeSquareFeet/DaysOnMarket']:
 0.9264


**Results**


***Non-Transform***

*Original* Highest R2 Score:  **0.7718**

*Enriched* Highest R2 Score:  **0.8099**

 *  Drastically Increased R2 Score

---
***Log Transform***

*Original* Highest R2 Score:  **0.8965**

*Enriched* Highest R2 Score:  **0.9264**
  
 *  Increased R2 Score

In [ ]:
reg_df = pd.DataFrame(regs)
reg_df

,Model,LogForm,ScoreType,ScoreValue,max_depth,learning_rate,n_estimators,columns
0,Linear Regression,False,R2 Score,0.710965,None,None,None,"[AvgAreaCost, LivingArea, BathroomsTotalIntege..."
1,Linear Regression,True,R2 Score,0.901440,None,None,None,"[AvgAreaCost, LivingArea, BathroomsTotalIntege..."
2,Decision Tree Regressor,False,R2 Score,0.708986,None,None,None,"[AvgAreaCost, AvgLivingArea, AvgAreaLot, Posta..."
3,Decision Tree Regressor,True,R2 Score,0.860948,None,None,None,"[AvgAreaCost, PostalCode, AvgLivingArea, AvgAr..."
4,Random Forest Regressor,False,R2 Score,0.809872,None,None,None,"[AvgAreaCost, AvgLivingArea, PostalCode, AvgAr..."
5,Random Forest Regressor,True,R2 Score,0.926433,None,None,None,"[AvgAreaCost, PostalCode, AvgLivingArea, AvgAr..."


In [ ]:
save_csv(reg_df, "regression_r2s.csv")